# Certamen 2 - Inteligencia de Negocios
## Caso 1: Mueblería Albarrán
### Modelo de Negocio, Exploración de Datos y Diseño Dimensional

Este notebook contiene el análisis exploratorio de la base de datos transaccional `Albarran`, la definición de KPIs para la toma de decisiones, la aplicación de los 4 pasos de Ralph Kimball y el diseño del modelo dimensional.

--- 
### 1. Conexión a SQL Server
La conexión se realiza directamente mediante SQLAlchemy y `pymssql` (compatible con macOS Apple Silicon ARM64 y Windows 11). 
Si estás en Docker deja `localhost:1433`. Si te conectas a la máquina Windows de un compañero, cambia `DB_HOST` por su IP de red.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlalchemy as sa

# Cargar credenciales estrictamente desde el archivo .env
root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(dotenv_path=root_dir / ".env")

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "1433")
DB_USER = os.getenv("DB_USER", "sa")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME", "Albarran")

if not DB_PASSWORD:
    raise ValueError("❌ Error: Variable DB_PASSWORD no definida en el archivo .env. Configúrala antes de continuar.")

# Construir engine de SQLAlchemy (sin contraseñas hardcodeadas)
connection_url = f"mssql+pymssql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = sa.create_engine(connection_url)

# Test de conexión rápida
with engine.connect() as conn:
    info = conn.execute(sa.text("SELECT @@VERSION AS ver, DB_NAME() AS current_db")).fetchone()
    print(f" Conectado exitosamente a la base '{info.current_db}' en {DB_HOST}:{DB_PORT}")
    print(f" Motor: {info.ver.splitlines()[0]}")


 Conectado exitosamente a la base 'Albarran' en localhost:1433
 Motor: Microsoft SQL Server 2022 (RTM-CU27-GDR) (KB5126934) - 16.0.4305.5 (X64) 


---
### 1.1 Utilidades de Reinicio y Purgado de Docker (macOS ARM64 / Linux)
Si durante el desarrollo o experimentación cometes un error en los datos o requieres volver a la versión limpia original:
- `restaurar_bd_docker()`: Realiza un `RESTORE` en caliente de `Albarran.bak` en solo 1 segundo sin reiniciar el contenedor.
- `reset_volumen_docker_completo()`: Detiene Docker, **purga el volumen por completo (`down -v`)**, levanta un contenedor nuevo y recarga el backup desde cero.


In [2]:
import subprocess
from pathlib import Path

def get_project_root() -> Path:
    current = Path.cwd()
    if (current / "docker" / "docker-compose.yml").exists():
        return current
    elif (current.parent / "docker" / "docker-compose.yml").exists():
        return current.parent
    return current

def restaurar_bd_docker():
    """Restaura en caliente la base de datos Albarran desde el backup en 1 segundo (sin reiniciar Docker)."""
    sql = """
    USE master;
    IF EXISTS (SELECT name FROM sys.databases WHERE name = 'Albarran')
    BEGIN
        ALTER DATABASE Albarran SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
        DROP DATABASE Albarran;
    END;
    RESTORE DATABASE Albarran
    FROM DISK = '/var/opt/mssql/backup/Albarran.bak'
    WITH MOVE 'Albarran1' TO '/var/opt/mssql/data/Albarran.mdf',
         MOVE 'Albarran1_log' TO '/var/opt/mssql/data/Albarran_log.ldf',
         REPLACE, RECOVERY;
    """
    with engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
        conn.execute(sa.text(sql))
    print(" Base de datos Albarran restaurada limpiamente desde el .bak")

def reset_volumen_docker_completo():
    """
    Elimina por completo el volumen de Docker (docker compose down -v),
    recrea el contenedor desde cero y restaura el backup limpio.
    """
    root = get_project_root()
    compose_file = root / "docker" / "docker-compose.yml"
    restore_script = root / "docker" / "restore.sh"
    
    print("🛑 Purgando contenedor y eliminando volumen de datos (docker compose down -v)...")
    subprocess.run(["docker", "compose", "-f", str(compose_file), "down", "-v"], check=True)
    
    print("🚀 Levantando contenedor limpio y restaurando backup...")
    subprocess.run([str(restore_script)], check=True)
    print("✨ Contenedor, volumen y base de datos restablecidos al 100%!")

# Para ejecutar el reseteo completo si ocurre un error, descomenta y corre:
# reset_volumen_docker_completo()
# O para solo recargar la base en caliente:
# restaurar_bd_docker()


--- 
### 2. Resumen de Tablas y Registros en Albarran
Consulta a los metadatos de SQL Server para comprobar las tablas restauradas desde `Albarran.bak`.

In [3]:
query_tablas = """
SELECT 
    t.name AS Tabla,
    SUM(p.rows) AS Total_Filas
FROM sys.tables t
JOIN sys.partitions p ON t.object_id = p.object_id AND p.index_id IN (0, 1)
GROUP BY t.name
ORDER BY t.name;
"""

with engine.connect() as conn:
    df_tablas = pd.read_sql(query_tablas, conn)

df_tablas

,Tabla,Total_Filas
0,cargos,5
1,categorias,13
2,clientes,20
3,comunas,3
4,detalle_ventas,365
5,empleados,15
6,productos,20
7,sucursales,3
8,ventas,365


--- 
### 3. Carga de Datos en Memoria para Exploración
Carga de las tablas transaccionales a DataFrames para iniciar el análisis.

In [4]:
with engine.connect() as conn:
    df_cargos = pd.read_sql('SELECT * FROM cargos', conn)
    df_categorias = pd.read_sql('SELECT * FROM categorias', conn)
    df_clientes = pd.read_sql('SELECT * FROM clientes', conn)
    df_comunas = pd.read_sql('SELECT * FROM comunas', conn)
    df_empleados = pd.read_sql('SELECT * FROM empleados', conn)
    df_productos = pd.read_sql('SELECT * FROM productos', conn)
    df_sucursales = pd.read_sql('SELECT * FROM sucursales', conn)
    df_ventas = pd.read_sql('SELECT * FROM ventas', conn)
    df_detalle = pd.read_sql('SELECT * FROM detalle_ventas', conn)

print(" Tablas cargadas en DataFrames.")

 Tablas cargadas en DataFrames.


--- 
### 4. Diagnóstico de Calidad de Datos (Data Quality)
*(Revisa aquí valores nulos, cardinalidades, inconsistencias en tipos de datos y reglas de negocio necesarias para el ETL)*

In [5]:
# Celda para análisis de nulos y calidad de datos


--- 
### 5. Definición y Justificación de KPIs
*(Define y justifica los KPIs que apoyarán al dueño de Mueblería Albarrán en la toma de decisiones: fórmula, utilidad, nivel de análisis)*

In [6]:
# Celda para cálculo y visualización preliminar de KPIs


--- 
### 6. Metodología de Kimball (4 Pasos)

1. **Elegir el proceso de negocio:** *(Ventas comerciales)*
2. **Establecer el nivel de granularidad:** *(Línea de detalle por producto vendido)*
3. **Elegir las dimensiones:** *(Dim_Cliente, Dim_Empleado, Dim_Producto, Dim_Sucursal, Dim_Tiempo)*
4. **Identificar medidas y tabla de hechos:** *(Fact_Ventas: Cantidad, Precio Unitario, Monto Bruto, Descuento, Monto Neto)*

In [7]:
# Celda para diseñar y simular el modelo dimensional
